# <font color="#418FDE" size="6.5" uppercase>**Spectral MDS TSNE**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply spectral embedding and MDS methods for nonlinear visualization. 
- Configure t-SNE perplexity, learning rate, initialization, and convergence settings. 
- Evaluate manifold embeddings using trustworthiness and visual comparison. 


## **1. Spectral MDS**

### **1.1. Spectral Embedding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_01_01.jpg?v=1784028815" width="250">



>* Builds neighbor graphs from high-dimensional data
>* Preserves nonlinear local relationships in visualization

>* Eigenvectors reveal smooth graph patterns
>* Maps clusters, bridges, and continuous structures

>* Graph choices shape nonlinear embeddings
>* Use exploratorily to reveal broader structure



In [ ]:
#@title Python Code - Spectral Embedding

# Demonstrate spectral embedding on curved synthetic data.
# Neighborhood graphs reveal nonlinear manifold structure.
# The plot shows unfolded coordinates by color.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import make_s_curve
from sklearn.manifold import SpectralEmbedding
from sklearn.manifold import trustworthiness

# Create a small curved dataset with known color ordering.
features, color_position = make_s_curve(n_samples=600, noise=0.05, random_state=42)

# Validate the dataset shape before fitting the embedding.
if features.shape != (600, 3):
    raise ValueError("Expected 600 samples with 3 features.")

# Fit spectral embedding using local nearest-neighbor connections.
embedding_model = SpectralEmbedding(
    n_components=2, n_neighbors=12, affinity="nearest_neighbors", random_state=42
)
embedded_points = embedding_model.fit_transform(features)

# Trustworthiness checks whether nearby embedded points were nearby originally.
score = trustworthiness(features, embedded_points, n_neighbors=12)

print("scikit-learn version:", sklearn_version)
print("Original shape:", features.shape)
print("Embedded shape:", embedded_points.shape)
print("Trustworthiness:", round(score, 3))

# Plot the two spectral coordinates colored by manifold position.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    embedded_points[:, 0], embedded_points[:, 1], c=color_position, s=18, cmap="viridis"
)

# Label the spectral coordinates for beginner interpretation.
ax.set_title("Spectral Embedding of an S-Curve Manifold")
ax.set_xlabel("Spectral coordinate 1")
ax.set_ylabel("Spectral coordinate 2")
fig.colorbar(scatter, ax=ax, label="Position along curved manifold")
plt.show()



### **1.2. Metric MDS**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_01_02.jpg?v=1784028817" width="250">



>* Turns pairwise distances into visual maps
>* Preserves closeness and separation between items

>* Turns distances into interpretable visual maps
>* Reveals clusters, gaps, and gradual transitions

>* Choose distances that match your data
>* Treat MDS maps as approximate visual summaries



In [ ]:
#@title Python Code - Metric MDS

# This example maps distances with metric MDS.
# Similar points should appear close together.
# The plot shows preserved distance relationships.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.manifold import MDS
from sklearn.metrics import pairwise_distances

# Create a small dataset with three visible groups.
features, labels = make_blobs(
    n_samples=90, centers=3, n_features=5, cluster_std=1.2, random_state=42
)

# Standardize features so distances compare variables fairly.
feature_means = features.mean(axis=0)
feature_stds = features.std(axis=0)
scaled_features = (features - feature_means) / feature_stds

# Metric MDS starts from pairwise distances between observations.
distances = pairwise_distances(scaled_features, metric="euclidean")
if distances.shape != (90, 90):
    raise ValueError("Expected a 90 by 90 distance table.")

# Fit one two-dimensional metric MDS embedding.
mds = MDS(
    n_components=2, metric=True, dissimilarity="precomputed", random_state=42
)
embedding = mds.fit_transform(distances)

# Compare original distances with distances in the map.
map_distances = pairwise_distances(embedding, metric="euclidean")
upper_triangle = np.triu_indices_from(distances, k=1)
correlation = np.corrcoef(
    distances[upper_triangle], map_distances[upper_triangle]
)[0, 1]

print("scikit-learn version:", sklearn.__version__)
print("Metric MDS stress:", round(float(mds.stress_), 2))
print("Distance correlation:", round(float(correlation), 3))

# Plot the two-dimensional map produced from distances.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    embedding[:, 0], embedding[:, 1], c=labels, cmap="viridis", s=45
)

ax.set_title("Metric MDS map from pairwise distances")
ax.set_xlabel("MDS dimension 1")
ax.set_ylabel("MDS dimension 2")
ax.legend(*scatter.legend_elements(), title="Group")
plt.show()



### **1.3. Nonmetric MDS**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_01_03.jpg?v=1784028819" width="250">



>* Preserves similarity rankings, not exact distances
>* Maps ordinal judgments into visual neighborhoods

>* Preserves rank order in low-dimensional maps
>* Handles subjective, uneven dissimilarity scales

>* Focus on clusters, not axis meanings
>* Validate exploratory maps with stress and context



In [ ]:
#@title Python Code - Nonmetric MDS

# Demonstrate nonmetric MDS with ordinal dissimilarities.
# Preserve rank order rather than exact distances.
# Compare original ranks with mapped distances visually.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS

# Create small clustered data with known neighborhood structure.
features, labels = make_blobs(
    n_samples=60, centers=3, n_features=5, cluster_std=1.2, random_state=42
)

# Convert exact distances into coarse ordinal dissimilarity ratings.
exact_distances = pairwise_distances(features)
ordinal_dissimilarities = np.ceil(exact_distances / exact_distances.max() * 5)
ordinal_dissimilarities = ordinal_dissimilarities.astype(float)

# Keep the diagonal at zero because each item matches itself.
np.fill_diagonal(ordinal_dissimilarities, 0.0)
if ordinal_dissimilarities.shape != (60, 60):
    raise ValueError("Expected a 60 by 60 dissimilarity matrix.")

# Fit nonmetric MDS to preserve the ordering of dissimilarities.
mds = MDS(
    n_components=2, metric=False, dissimilarity="precomputed", random_state=42,
    normalized_stress="auto", n_init=4, max_iter=300, eps=0.001
)
embedding = mds.fit_transform(ordinal_dissimilarities)

# Measure how well mapped distances follow ordinal dissimilarity ranks.
mapped_distances = pairwise_distances(embedding)
upper_triangle = np.triu_indices_from(ordinal_dissimilarities, k=1)
rank_correlation = np.corrcoef(
    ordinal_dissimilarities[upper_triangle], mapped_distances[upper_triangle]
)[0, 1]

print("scikit-learn version:", sklearn.__version__)
print("Nonmetric MDS stress:", round(float(mds.stress_), 3))
print("Rank-distance correlation:", round(float(rank_correlation), 3))

# Plot the two-dimensional nonmetric MDS map.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    embedding[:, 0], embedding[:, 1], c=labels, cmap="viridis", s=55
)

ax.set_title("Nonmetric MDS from ordinal dissimilarities")
ax.set_xlabel("MDS dimension 1")
ax.set_ylabel("MDS dimension 2")
ax.legend(*scatter.legend_elements(), title="Cluster")
plt.show()



## **2. TSNE Controls**

### **2.1. SMACOF Optimization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_02_01.jpg?v=1784028823" width="250">



>* SMACOF lowers MDS stress step by step
>* Initialization and convergence affect embedding results

>* SMACOF adjusts layouts to match dissimilarities
>* Different starts can yield different stable maps

>* Tune starts, updates, and stopping carefully
>* Rerun embeddings and compare stability



### **2.2. t SNE Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_02_02.jpg?v=1784028821" width="250">



>* Maps high-dimensional data for visual exploration
>* Keeps similar observations close together

>* Matches high-dimensional and map neighborhoods
>* Heavy tails separate clusters, but interpret cautiously

>* Places similar items near each other
>* Use cautiously; settings affect the map



In [ ]:
#@title Python Code - t SNE Basics

# This example introduces basic t-SNE controls.
# Perplexity changes the neighborhood scale emphasized.
# The plot compares two deterministic t-SNE maps.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Load a small packaged digit image dataset.
digits = load_digits()

# Use a small deterministic subset for quick visualization.
sample_count = 300
features = digits.data[:sample_count]
labels = digits.target[:sample_count]

# Validate the expected two-dimensional feature table.
if features.ndim != 2 or len(labels) != sample_count:
    raise ValueError("Expected a two-dimensional feature table.")

# Standardize features so pixel scales are comparable.
scaled_features = StandardScaler().fit_transform(features)

# Configure two t-SNE runs with different perplexity values.
low_perplexity = TSNE(
    n_components=2,
    perplexity=5,
    learning_rate="auto",
    init="pca",
    max_iter=750,
    random_state=42,
)

high_perplexity = TSNE(
    n_components=2,
    perplexity=40,
    learning_rate="auto",
    init="pca",
    max_iter=750,
    random_state=42,
)

# Fit both maps to compare local neighborhood settings.
low_map = low_perplexity.fit_transform(scaled_features)
high_map = high_perplexity.fit_transform(scaled_features)

# Shift the second map right for one combined comparison plot.
gap = 8.0
high_map_shifted = high_map.copy()
high_map_shifted[:, 0] = high_map_shifted[:, 0] + gap

print("scikit-learn version:", sklearn.__version__)
print("Dataset: sklearn load_digits subset with", sample_count, "samples")
print("Left map perplexity: 5, right map perplexity: 40")
print("Both runs use learning_rate='auto', init='pca', max_iter=750")

# Draw both embeddings on one axes for a focused comparison.
fig, ax = plt.subplots(figsize=(8, 5))
scatter_left = ax.scatter(
    low_map[:, 0],
    low_map[:, 1],
    c=labels,
    cmap="tab10",
    s=18,
    alpha=0.8,
)

ax.scatter(
    high_map_shifted[:, 0],
    high_map_shifted[:, 1],
    c=labels,
    cmap="tab10",
    s=18,
    alpha=0.8,
)

ax.axvline(gap / 2, color="gray", linestyle="--", linewidth=1)
ax.text(np.min(low_map[:, 0]), np.max(low_map[:, 1]), "perplexity = 5")
ax.text(np.min(high_map_shifted[:, 0]), np.max(high_map[:, 1]), "perplexity = 40")
ax.set_title("t-SNE basics: perplexity changes neighborhood emphasis")

ax.set_xlabel("t-SNE dimension 1, shifted for comparison")
ax.set_ylabel("t-SNE dimension 2")
legend = ax.legend(*scatter_left.legend_elements(), title="Digit", loc="best")
ax.add_artist(legend)
plt.show(block=False)
plt.pause(0.001)



### **2.3. Perplexity Learning Rate**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_02_03.jpg?v=1784028825" width="250">



>* Perplexity sets each point’s neighborhood scale
>* Low values reveal detail; high values smooth

>* Learning rate controls t-SNE update speed
>* Compare runs to avoid misleading embeddings

>* Tune perplexity and learning rate together
>* Trust stable local patterns across runs



In [ ]:
#@title Python Code - Perplexity Learning Rate

# Compare t-SNE settings on handwritten digit data.
# Perplexity and learning rate change neighborhood preservation.
# Trustworthiness scores summarize each embedding quality.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
from sklearn.preprocessing import StandardScaler

# Load a small packaged dataset for fast visualization.
digits = load_digits()
features = digits.data[:600]
labels = digits.target[:600]

# Validate the basic shape before running t-SNE.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Scale pixel features so distances are more balanced.
scaled_features = StandardScaler().fit_transform(features)

# Try three focused t-SNE control settings.
settings = [
    {"perplexity": 5, "learning_rate": 50},
    {"perplexity": 30, "learning_rate": 200},
    {"perplexity": 50, "learning_rate": 500},
]

scores = []
embeddings = []

# Fit one t-SNE embedding for each setting.
for setting in settings:
    tsne = TSNE(
        n_components=2,
        perplexity=setting["perplexity"],
        learning_rate=setting["learning_rate"],
        init="pca",
        max_iter=750,
        random_state=42,
    )
    embedding = tsne.fit_transform(scaled_features)
    score = trustworthiness(scaled_features, embedding, n_neighbors=10)
    embeddings.append(embedding)
    scores.append(score)

# Select the setting with the highest trustworthiness score.
best_index = int(np.argmax(scores))
best_setting = settings[best_index]
best_embedding = embeddings[best_index]

print(f"scikit-learn version: {sklearn_version}")
print("t-SNE settings compared with trustworthiness:")

# Print concise results for beginner comparison.
for setting, score in zip(settings, scores):
    p = setting["perplexity"]
    lr = setting["learning_rate"]
    print(f"perplexity={p}, learning_rate={lr}: trust={score:.3f}")

# Plot the best-scoring embedding from the small comparison.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    best_embedding[:, 0],
    best_embedding[:, 1],
    c=labels,
    cmap="tab10",
    s=18,
    alpha=0.85,
)

ax.set_title(
    f"Best t-SNE: perplexity={best_setting['perplexity']}, "
    f"learning rate={best_setting['learning_rate']}"
)
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")

legend = ax.legend(
    *scatter.legend_elements(),
    title="Digit",
    loc="best",
    fontsize=8,
)
ax.add_artist(legend)
plt.show()



## **3. Embedding Evaluation**

### **3.1. Initialization Stability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_03_01.jpg?v=1784028826" width="250">



>* Different starts can change nonlinear embeddings
>* Stable patterns increase confidence in results

>* Rerun embeddings with varied starting points
>* Trust patterns that consistently reappear

>* Repeated runs reveal stable versus fragile patterns
>* Interpret recurring neighborhoods with cautious supporting evidence



In [ ]:
#@title Python Code - Initialization Stability

# This example checks t-SNE initialization stability.
# Repeated runs should preserve trusted neighborhoods.
# Trustworthiness summarizes stability beyond visual appearance.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
from sklearn.preprocessing import StandardScaler

# Load a small built-in image dataset.
digits = load_digits()

# Use a deterministic subset for fast repeated embeddings.
selected = np.arange(300)
features = digits.data[selected]
labels = digits.target[selected]

# Validate the expected two-dimensional table shape.
if features.ndim != 2 or features.shape[0] != labels.shape[0]:
    raise ValueError("Features and labels must align by row.")

# Scale features before measuring neighborhoods.
scaled_features = StandardScaler().fit_transform(features)

# Compare two different random initializations.
embedding_a = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="random",
    max_iter=750,
    random_state=42,
).fit_transform(scaled_features)

embedding_b = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="random",
    max_iter=750,
    random_state=7,
).fit_transform(scaled_features)

# Trustworthiness checks whether original neighbors stay nearby.
trust_a = trustworthiness(scaled_features, embedding_a, n_neighbors=10)
trust_b = trustworthiness(scaled_features, embedding_b, n_neighbors=10)

print("scikit-learn version:", sklearn.__version__)
print("Run A trustworthiness:", round(trust_a, 3))
print("Run B trustworthiness:", round(trust_b, 3))
print("Stable interpretation: compare recurring digit neighborhoods, not axes.")

# Plot one run and use color to inspect neighborhood structure.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    embedding_a[:, 0],
    embedding_a[:, 1],
    c=labels,
    cmap="tab10",
    s=28,
    alpha=0.85,
)

ax.set_title("t-SNE run A: inspect recurring digit neighborhoods")
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
legend = ax.legend(*scatter.legend_elements(), title="Digit", loc="best")
ax.add_artist(legend)
plt.show(block=False)
plt.pause(0.001)
plt.close(fig)



### **3.2. Trustworthiness Score**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_03_02.jpg?v=1784028830" width="250">



>* Checks whether nearby embedded points were truly similar
>* High scores reduce misleading visual neighborhood artifacts

>* Higher scores mean fewer false local neighbors
>* Choose neighborhood size based on analysis goals

>* Trustworthiness is useful but incomplete.
>* Combine with visuals, expertise, and stability checks.



In [ ]:
#@title Python Code - Trustworthiness Score

# Compare embeddings using a trustworthiness score.
# Trustworthiness checks whether local neighbors stay reliable.
# Higher scores support more trustworthy visual neighborhoods.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.manifold import MDS
from sklearn.manifold import trustworthiness
from sklearn.preprocessing import StandardScaler

# Load a small built-in handwritten digits dataset.
digits = load_digits()
features = digits.data[:300]
labels = digits.target[:300]

# Validate the basic shape before embedding.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Feature rows and labels must match.")

# Scale features so pixel ranges are comparable.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Create one two-dimensional embedding for visualization.
mds = MDS(n_components=2, random_state=42, normalized_stress="auto")
embedding = mds.fit_transform(scaled_features)

# Measure whether nearby embedded points were originally nearby.
score_5 = trustworthiness(scaled_features, embedding, n_neighbors=5)
score_20 = trustworthiness(scaled_features, embedding, n_neighbors=20)

print("scikit-learn version:", sklearn.__version__)
print("Trustworthiness with 5 neighbors:", round(score_5, 3))
print("Trustworthiness with 20 neighbors:", round(score_20, 3))

# Plot the embedding so the score can be compared visually.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap="tab10", s=25)

ax.set_title("MDS embedding of handwritten digits")
ax.set_xlabel("Embedding dimension 1")
ax.set_ylabel("Embedding dimension 2")

legend = ax.legend(*scatter.legend_elements(), title="Digit", loc="best")
ax.add_artist(legend)
plt.show()



### **3.3. Method Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_B/image_03_03.jpg?v=1784028828" width="250">



>* Compare what each embedding method preserves
>* Match visual patterns to analysis goals

>* Combine trustworthiness scores with visual checks
>* Treat inconsistent clusters as tentative hypotheses

>* Compare embeddings under consistent preprocessing
>* Trust stable patterns, investigate method differences



In [ ]:
#@title Python Code - Method Comparison

# Compare embeddings with one neighborhood metric.
# Trustworthiness checks local structure preservation.
# The plot shows visual method differences.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.manifold import MDS
from sklearn.manifold import SpectralEmbedding
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
from sklearn.preprocessing import StandardScaler

# Load a small built-in handwritten digits dataset.
digits = load_digits()
features = digits.data[:300]
labels = digits.target[:300]

# Validate the subset before fitting embeddings.
if features.shape != (300, 64):
    raise ValueError("Expected 300 digit images with 64 pixel features.")

# Scale features so distance-based methods compare pixels fairly.
scaled_features = StandardScaler().fit_transform(features)

# Build three embeddings with comparable two-dimensional output.
spectral = SpectralEmbedding(n_components=2, random_state=42)
mds = MDS(n_components=2, random_state=42, normalized_stress="auto", n_init=1)
tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42, max_iter=750)

# Fit each method on the same preprocessed data.
spectral_points = spectral.fit_transform(scaled_features)
mds_points = mds.fit_transform(scaled_features)
tsne_points = tsne.fit_transform(scaled_features)

# Trustworthiness compares original neighbors with embedded neighbors.
spectral_score = trustworthiness(scaled_features, spectral_points, n_neighbors=10)
mds_score = trustworthiness(scaled_features, mds_points, n_neighbors=10)
tsne_score = trustworthiness(scaled_features, tsne_points, n_neighbors=10)

print("scikit-learn version:", sklearn.__version__)
print("Trustworthiness, Spectral:", round(spectral_score, 3))
print("Trustworthiness, MDS:", round(mds_score, 3))
print("Trustworthiness, t-SNE:", round(tsne_score, 3))

# Plot trustworthiness beside a simple visual spread measure.
method_names = ["Spectral", "MDS", "t-SNE"]
trust_scores = [spectral_score, mds_score, tsne_score]

# Normalize visual spread for a compact comparison chart.
spreads = []
for points in [spectral_points, mds_points, tsne_points]:
    spread = np.mean(np.std(points, axis=0))
    spreads.append(spread)

scaled_spreads = np.array(spreads) / max(spreads)
x_positions = np.arange(len(method_names))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x_positions - 0.18, trust_scores, width=0.36, label="Trustworthiness")
ax.bar(x_positions + 0.18, scaled_spreads, width=0.36, label="Relative spread")
ax.set_xticks(x_positions)
ax.set_xticklabels(method_names)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_xlabel("Embedding method")
ax.set_title("Embedding comparison: local trust and visual spread")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Spectral MDS TSNE**</font>


In this lecture, you learned to:
- Apply spectral embedding and MDS methods for nonlinear visualization. 
- Configure t-SNE perplexity, learning rate, initialization, and convergence settings. 
- Evaluate manifold embeddings using trustworthiness and visual comparison. 

In the next Module (Module 26), we will go over 'Mixtures Outliers'